## Data Ingestion Lab

In [53]:
!uv pip install chroma langchain langchain-openai langchain-text-splitters langchain-chroma langchain-community pyyaml

Using Python 3.12.12 environment at: /Users/davidinyang-etoh/Projects/ai-projects/ai-playground/.venv
Audited 7 packages in 68ms


### Import modules and packages

In [54]:
import os
import re
import yaml
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownTextSplitter

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [64]:
MODEL = "gpt-4.1-nano"

EMBEDDING_MODEL = "text-embedding-3-small"

# Larger chunks so sections stay whole; more overlap for context continuity
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [56]:
DOMAIN = "topfaith.edu.ng"

DB_NAME = str(Path("db/vector_db"))
KNOWLEDGE_BASE_PATH = str(f"data/{DOMAIN}")

print(f"Loading documents from {KNOWLEDGE_BASE_PATH}")

Loading documents from data/topfaith.edu.ng


### Pre-processing: frontmatter parsing + boilerplate cleaning

In [57]:
from datetime import date

# Residual boilerplate patterns that survive the crawler's HTML cleaning
_BOILERPLATE = re.compile(
    r'^Email Us\s+Call us today\s*$'       # contact bar left in body divs
    r'|©\s*\d{4}\s+Topfaith University\.?' # copyright footer
    r'|\bView Fees?(?: Breakdown)?\b'       # fees link text
    r'|\bExplore Now\s*→?\s*',
    re.IGNORECASE | re.MULTILINE,
)


def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Strip YAML frontmatter from markdown. Returns (metadata_dict, body)."""
    if not text.startswith("---\n"):
        return {}, text
    end = text.find("\n---", 4)
    if end == -1:
        return {}, text
    try:
        metadata = yaml.safe_load(text[4:end]) or {}
    except yaml.YAMLError:
        metadata = {}
    return metadata, text[end + 4:].lstrip("\n")


def clean_body(text: str) -> str:
    """Remove residual boilerplate lines and collapse extra whitespace."""
    text = _BOILERPLATE.sub("", text)
    # Collapse runs of blank lines to a single blank line
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _chroma_friendly_scalar(v):
    """Chroma metadata must be str/int/float/bool/list/None — YAML timestamps become datetime."""
    if v is None:
        return ""
    if isinstance(v, date):
        return v.isoformat()
    return v


def preprocess_documents(documents):
    """
    For each loaded document:
      1. Parse YAML frontmatter → LangChain metadata fields
      2. Replace page_content with cleaned body (no frontmatter, no boilerplate)
    """
    for doc in documents:
        fm, body = parse_frontmatter(doc.page_content)
        if fm:
            doc.metadata.update({
                "url":           fm.get("url", ""),
                "page_title":    fm.get("title", ""),
                "breadcrumb":    fm.get("breadcrumb", ""),
                # Chroma requires scalar values — join list as comma string
                "path_segments": ",".join(fm.get("path_segments") or []),
                "crawled_at":    _chroma_friendly_scalar(fm.get("crawled_at", "")),
            })
        doc.page_content = clean_body(body or doc.page_content)
    return documents

In [58]:
def fetch_documents():
    folders = glob.glob(str(Path(KNOWLEDGE_BASE_PATH)))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

documents = fetch_documents()

print(f"Found {len(documents)} documents in {KNOWLEDGE_BASE_PATH}")

Found 42 documents in data/topfaith.edu.ng


In [59]:
def create_chunks(documents):
    text_splitter = MarkdownTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = text_splitter.split_documents(documents)

    enriched = []
    for chunk in chunks:
        # Drop near-empty chunks (e.g. a heading with no body text)
        if len(chunk.page_content.strip()) < 60:
            continue

        # Prepend breadcrumb as context so the embedding captures page location.
        # Falls back to page_title, then doc_type if breadcrumb is absent.
        breadcrumb = (
            chunk.metadata.get("breadcrumb")
            or chunk.metadata.get("page_title")
            or chunk.metadata.get("doc_type", "")
        )
        if breadcrumb:
            chunk.page_content = f"[{breadcrumb}]\n\n{chunk.page_content}"

        enriched.append(chunk)

    return enriched

In [60]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME,
               embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])[
        "embeddings"][0]
    dimensions = len(sample_embedding)
    print(
        f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [61]:
documents = fetch_documents()
documents = preprocess_documents(documents)
chunks = create_chunks(documents)
create_embeddings(chunks)
print("Ingestion complete")

There are 386 vectors with 1,536 dimensions in the vector store
Ingestion complete


In [62]:
def get_chunks(query):
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=DB_NAME
    )
    return vectorstore.similarity_search(query)


results = get_chunks("Fees")

print(results)

[Document(id='f84bafa1-cd64-4cee-adb9-41a83a1df052', metadata={'source': 'data/topfaith.edu.ng/pg_fees.md', 'doc_type': 'topfaith.edu.ng', 'breadcrumb': 'Pg > Fees', 'crawled_at': '2026-03-29T14:13:06.859610', 'page_title': 'Fees - Topfaith University', 'path_segments': 'pg,fees', 'url': 'https://www.topfaith.edu.ng/pg/fees'}, page_content='[Pg > Fees]\n\n# Fees - Topfaith University\n\n## Fees\n\n## 2025/2026 SESSION FEES FOR FRESHERS:'), Document(id='3a3af315-2117-40da-ac51-4a7d92971c08', metadata={'source': 'data/topfaith.edu.ng/pg_fees.md', 'doc_type': 'topfaith.edu.ng', 'page_title': 'Fees - Topfaith University', 'url': 'https://www.topfaith.edu.ng/pg/fees', 'breadcrumb': 'Pg > Fees', 'path_segments': 'pg,fees', 'crawled_at': '2026-03-29T14:13:06.859610'}, page_content='[Pg > Fees]\n\nI) ACCEPTANCE FEE:Non-Refundable acceptance fee (Freshers Only) II) TUITION, ACCOMMODATION & OTHER FEES (covering Registration, Tutorials, ICT, Practicals & General Studies Materials): | MANAGEMENT &

In [63]:
from IPython.display import Markdown
chunk = results[0]

print(len(results))

chunk_count = 1;
for result in results:
    print(f"No: {chunk_count}")
    print("="*100)
    print(result.metadata)
    print("-"*100)
    display(Markdown(result.page_content))
    chunk_count += 1

4
No: 1
{'source': 'data/topfaith.edu.ng/pg_fees.md', 'doc_type': 'topfaith.edu.ng', 'breadcrumb': 'Pg > Fees', 'crawled_at': '2026-03-29T14:13:06.859610', 'page_title': 'Fees - Topfaith University', 'path_segments': 'pg,fees', 'url': 'https://www.topfaith.edu.ng/pg/fees'}
----------------------------------------------------------------------------------------------------


[Pg > Fees]

# Fees - Topfaith University

## Fees

## 2025/2026 SESSION FEES FOR FRESHERS:

No: 2
{'source': 'data/topfaith.edu.ng/pg_fees.md', 'doc_type': 'topfaith.edu.ng', 'page_title': 'Fees - Topfaith University', 'url': 'https://www.topfaith.edu.ng/pg/fees', 'breadcrumb': 'Pg > Fees', 'path_segments': 'pg,fees', 'crawled_at': '2026-03-29T14:13:06.859610'}
----------------------------------------------------------------------------------------------------


[Pg > Fees]

I) ACCEPTANCE FEE:Non-Refundable acceptance fee (Freshers Only) II) TUITION, ACCOMMODATION & OTHER FEES (covering Registration, Tutorials, ICT, Practicals & General Studies Materials): | MANAGEMENT & SOCIAL SCIENCES:Accounting (4 years)Business Administration (4 years)Criminology and Security Studies (4 years)Economics (4 years)Mass Communication (4 years) | N 2,000,000N 2,000,000N 2,000,000N 2,000,000N 2,000,000 | COMPUTING & APPLIED SCIENCES:Architecture (4 years)Biotechnology (4

No: 3
{'breadcrumb': 'Pg > Fees', 'page_title': 'Fees - Topfaith University', 'crawled_at': '2026-03-29T14:13:06.859610', 'doc_type': 'topfaith.edu.ng', 'source': 'data/topfaith.edu.ng/pg_fees.md', 'path_segments': 'pg,fees', 'url': 'https://www.topfaith.edu.ng/pg/fees'}
----------------------------------------------------------------------------------------------------


[Pg > Fees]

## SCHOLARSHIP AND AWARDS

In line with our core value of excellence, the following scholarship opportunities are available:

### CATEGORY A- ENTRY SCHOLARSHIP

No: 4
{'breadcrumb': 'Pg > Fees', 'page_title': 'Fees - Topfaith University', 'source': 'data/topfaith.edu.ng/pg_fees.md', 'doc_type': 'topfaith.edu.ng', 'crawled_at': '2026-03-29T14:13:06.859610', 'path_segments': 'pg,fees', 'url': 'https://www.topfaith.edu.ng/pg/fees'}
----------------------------------------------------------------------------------------------------


[Pg > Fees]

Dean's list award: 5% tuition fee discount for students with 1st class Cumulative Grade Point Average (CGPA) at the end of session. BOT Academic Excellence award: 10% tuition fee discount for students with a 5.0 Cumulative Grade Point Average (CGPA) at the end of session. Thomas Abraham foundation valedictorian award of One million Naira (N 1,000,000) for the overall best graduating student

### CATEGORY C- MULTIPLE SIBLING DISCOUNT